# RAGAS Benchmark — Sherlock Holmes Corpus

This notebook runs the **RAGAS quality evaluation** on RAG methods using the Sherlock Holmes corpus.

**Dataset**: Adventures of Sherlock Holmes (full text) chunked into overlapping passages.
Test questions are generated via `scripts/generate_testset.py` and evaluated on 4 RAGAS metrics:
- **Faithfulness** — Is the answer grounded in the retrieved context?
- **Answer Relevancy** — Does the answer address the question?
- **Context Precision** — Are the retrieved chunks relevant to the question?
- **Context Recall** — Do the retrieved chunks cover the ground truth?

In [1]:
# --- Environment setup (must run first) ---
from nb_helpers.config import setup_environment, load_config
setup_environment()

from helpers.types import BenchmarkConfig

# --- Configuration ---
# Edit these variables before running

METHODS = ["vdb"]                  # Adapter methods to benchmark
MAX_QUESTIONS = 10                  # Number of test questions (0 = all)
TOP_K = 8                           # Chunks to retrieve per query
CHUNK_SIZE = 1200                   # Characters per chunk
CHUNK_OVERLAP = 200                 # Overlap between chunks
SKIP_INDEX = False                  # Set True to reuse existing indexes

config_dict = load_config(overrides={
    "methods": METHODS,
    "max_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})
config = BenchmarkConfig.from_dict(config_dict)
print(f"Config: {len(METHODS)} methods, {MAX_QUESTIONS} questions, top_k={TOP_K}")

Config: 1 methods, 10 questions, top_k=8


In [2]:
# --- Download corpus (if not already present) ---
import subprocess
import os
from nb_helpers.config import PROJECT_ROOT

corpus_file = PROJECT_ROOT / "corpus" / "adventures_of_sherlock_holmes.txt"
if not corpus_file.exists():
    print("Corpus not found — downloading from Project Gutenberg...")
    script = PROJECT_ROOT / "scripts" / "download_corpus.sh"
    subprocess.run(["bash", str(script)], check=True)
    print()
else:
    print(f"Corpus already present: {corpus_file}")

# Check test set
testset_file = PROJECT_ROOT / "testset" / "sherlock_holmes.json"
if not testset_file.exists():
    print("\nTest set not found. Generate it with:")
    print("  python scripts/generate_testset.py")
else:
    print(f"Test set present: {testset_file}")

Corpus not found — downloading from Project Gutenberg...
Stripping Gutenberg header and footer...
Done: /Users/enamakel/work/bnbpad.ai/neocortex-docs/mk1/corpus/adventures_of_sherlock_holmes.txt (   11932 lines,   104506 words)

Test set present: /Users/enamakel/work/bnbpad.ai/neocortex-docs/mk1/testset/sherlock_holmes.json


In [3]:
# --- Load Dataset ---
from nb_helpers.datasets import load_ragas_sherlock

chunks, testset = load_ragas_sherlock(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    max_questions=MAX_QUESTIONS,
)
print(f"\nSample question: {testset[0]['question']}")
print(f"Ground truth: {testset[0]['ground_truth'][:200]}...")

Sherlock Holmes corpus: 562,213 chars -> 563 chunks
Test set: 10 questions

Sample question: In both 'The Adventure of the Beryl Coronet' and 'A Case of Identity', how does Sherlock Holmes employ disguise or deception, and what does this reveal about his investigative approach?
Ground truth: In 'The Adventure of the Beryl Coronet', Holmes disguises himself as a common loafer, which illustrates his willingness to immerse himself into different social situations to gather information direct...


In [ ]:
# --- Run Benchmarks ---
from nb_helpers.pipeline import run_all_methods

all_results = await run_all_methods(
    methods=METHODS,
    chunks=chunks,
    testset=testset,
    config=config,
    skip_index=SKIP_INDEX,
)

Running 1 methods: vdb

--- vdb (1/1) ---
  [vdb] Indexing 563 chunks...


/Users/enamakel/work/bnbpad.ai/fast-graphrag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/enamakel/work/bnbpad.ai/fast-graphrag/.venv/lib/python3.13/site-packages/instructor/client_gemini.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


  [vdb] Indexed in 5.5s ($0.0035)
  [vdb] Querying 10 questions...
  [vdb] 5/10 questions done


In [ ]:
# --- Evaluate with RAGAS ---
from nb_helpers.metrics import evaluate_all_methods

all_results = await evaluate_all_methods(
    all_results,
    config=config,
    use_ragas=True,
    use_em_f1=False,
)

In [ ]:
# --- Summary Table ---
from nb_helpers.charts import summary_table

summary_table(all_results)

In [ ]:
# --- Charts ---
from nb_helpers.charts import ragas_bar_chart, latency_chart, cost_breakdown_chart

ragas_bar_chart(all_results, title="RAGAS Scores — Sherlock Holmes")
latency_chart(all_results, title="Avg Query Latency — Sherlock Holmes")
cost_breakdown_chart(all_results, title="Cost Breakdown — Sherlock Holmes")

In [ ]:
# --- Save Results ---
import json, os
from nb_helpers.config import PROJECT_ROOT
from datetime import datetime, timezone

output_dir = PROJECT_ROOT / "results" / "notebook_ragas"
os.makedirs(output_dir, exist_ok=True)

for method_name, method_data in all_results.items():
    filepath = output_dir / f"{method_name}.json"
    payload = {
        "run_config": {
            "corpus": "sherlock_holmes",
            "num_questions": len(testset),
            "top_k": TOP_K,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        },
        "method": method_name,
        **method_data,
    }
    with open(filepath, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved: {filepath}")